In [1]:
"""Train the OCSVM detector and scale its anomaly scores."""

import os
import sys

import pandas as pd
from pyod.models.ocsvm import OCSVM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
ocsvm = OCSVM(
    contamination=config.CONTAMINATION,
    kernel=config.KERNEL,
    nu=config.NU,
    gamma=config.GAMMA,
    degree=config.DEGREE,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
ocsvm.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = ocsvm.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    ocsvm.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="ocsvm",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


2026-08-14 18:17:50.369003: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


2026-08-14 18:29:27.223004: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.0105 - val_loss: 0.0082
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0074 - val_loss: 0.0077
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0069 - val_loss: 0.0071
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0067 - val_loss: 0.0069
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.0066 - val_loss: 0.0066
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0066 - val_loss: 0.0066
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.0065 - val_loss: 0.0069
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0058 - val_loss: 0.0047
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0044 - val_loss: 0.0042
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0043 - val_loss: 0.0047
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0042 - val_loss: 0.0042
Epoch 12/50
3747/3747 ━━━━━━━━━━━━━━━━━

INFO:tensorflow:Assets written to: /tmp/tmprfsojqty/assets


Saved artifact at '/tmp/tmprfsojqty'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139873231964304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139873231967952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139873231966608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139873231965264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139873231968528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139873231965456: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786725349.598310   20120 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786725349.598335   20120 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:35:49.598689: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmprfsojqty
2026-08-14 18:35:49.599030: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:35:49.599035: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmprfsojqty
I0000 00:00:1786725349.603210   20120 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:35:49.603922: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>